# 3.1 — Time-Frequency Analysis

This notebook covers frequency-domain and time-frequency analysis of OPM-MEG
data, adapted from the MNE tutorials:
- [The Spectrum and EpochsSpectrum classes](https://mne.tools/stable/auto_tutorials/time-freq/10_spectrum_class.html)
- [Frequency and time-frequency sensor analysis](https://mne.tools/stable/auto_tutorials/time-freq/20_sensors_time_frequency.html)

We load the cleaned epochs directly from the file saved by the preprocessing
pipeline and work from there — no need to recompute anything from raw data.

**What we will cover:**
1. **Power Spectral Density (PSD)** — `epochs.compute_psd()`, visualising the
   spectrum, spatial distribution via topomap
2. **Mean vs median PSD** — robustness considerations
3. **Time-frequency representations (TFR)** — Morlet wavelets via
   `mne.time_frequency.tfr_morlet()`, induced power and inter-trial coherence (ITC)
4. **TFR visualisation** — `power.plot_topo()`, `power.plot_joint()`
5. **ITC visualisation** — `itc.plot_topo()`

---
### Running this notebook

```bash
uv run --project $MNE_OPM_DIR jupyter lab
```

Use `%matplotlib inline` for static figures (default below).  
Switch to `%matplotlib qt` for interactive figures.


## 0. Setup


In [ ]:
%matplotlib inline
# %matplotlib qt  # uncomment for interactive plots

import pathlib
import numpy as np
import matplotlib.pyplot as plt
import mne

mne.set_log_level('WARNING')


In [ ]:
# ─── USER SETTINGS ───────────────────────────────────────────────────────────
DERIV_DIR = pathlib.Path('/your/path/to/sub-001/ses-01/meg')

SUBJECT   = 'sub-001'
SESSION   = 'ses-01'
TASK      = 'oddball'

STANDARD_COND = 'standard_onset'
DEVIANT_COND  = 'deviant_onset'
# ─────────────────────────────────────────────────────────────────────────────

epo_file = DERIV_DIR / f'{SUBJECT}_{SESSION}_task-{TASK}_proc-clean_epo.fif'
print(f'Epochs file : {epo_file}')
print(f'File exists : {epo_file.exists()}')


## 1. Load Epochs

We load the preprocessed, cleaned epochs directly. All time-frequency analyses
in this notebook operate on the trial-level data — not on the averaged Evoked
objects — because TFR and PSD require access to individual trials.


In [ ]:
epochs = mne.read_epochs(epo_file, preload=True, verbose=False)

# Select Z-component magnetometer channels only, excluding bads.
# Z channels are the radial component — most comparable to traditional SQUID-MEG.
epochs_z = epochs.copy().pick('mag', exclude='bads')
z_names  = [ch for ch in epochs_z.ch_names if ch.endswith(' Z')]
epochs_z = epochs_z.pick_channels(z_names)

print(epochs_z)
print(f'\nZ channels : {len(z_names)}')
print(f'Conditions : {list(epochs_z.event_id.keys())}')
print(f'Sfreq      : {epochs_z.info["sfreq"]} Hz')


## 2. Power Spectral Density

The **Power Spectral Density (PSD)** describes how signal power is distributed
across frequencies. For MEG data it reveals:
- The 1/f background noise floor (pink noise)
- Line noise peaks (60 Hz)
- Neural oscillations (alpha ~10 Hz, beta ~20 Hz, gamma >30 Hz)

`epochs.compute_psd()` returns an `EpochsSpectrum` object. Under the hood it
uses Welch's method: each epoch is divided into overlapping windows, the
periodogram is computed for each window, and the results are averaged. The
`fmin` / `fmax` arguments restrict the frequency range.


In [ ]:
# compute_psd() applies Welch's method to each epoch.
# fmin/fmax restrict the output frequency range.
# picks='mag' ensures we use only magnetometer channels.
psd = epochs_z.compute_psd(
    method='welch',
    fmin=1.0,
    fmax=100.0,
    picks='mag',
    verbose=False,
)
print(psd)
print(f'Frequency resolution : {psd.freqs[1] - psd.freqs[0]:.3f} Hz')
print(f'Frequency range      : {psd.freqs[0]:.1f} – {psd.freqs[-1]:.1f} Hz')


### 2a. Visualising the spectrum

`psd.plot()` averages across epochs and channels to give a single summary
spectrum. The shaded region shows the inter-channel variability.
Peaks above the 1/f floor indicate oscillatory activity or artifacts.


In [ ]:
# plot() averages across epochs and shows one line per channel type.
# amplitude=False plots power (µT²/Hz or T²/Hz); True plots amplitude spectral density.
fig = psd.plot(
    picks='mag',
    amplitude=False,
    show=False,
)
plt.show()


### 2b. Spatial distribution of power — topomap

`psd.plot_topomap()` shows how power is spatially distributed across sensors
for a given frequency band. This is useful for identifying which brain regions
contribute most to a given oscillation, or for spotting spatially localized
artifacts.

We define five standard frequency bands and pass them to `bands=` as a dict
of `{label: (fmin, fmax)}` pairs:

- **Delta** (1–4 Hz) — slow oscillations; at sensor level often dominated by
  low-frequency noise and movement artifacts
- **Theta** (4–8 Hz) — memory encoding, cognitive load; prominent over
  frontal sensors
- **Alpha** (8–12 Hz) — idling / attention; prominent over posterior sensors
  in resting state, suppressed during auditory stimulation
- **Beta** (15–30 Hz) — motor and somatosensory cortex; event-related
  desynchronisation during movement or sensory processing
- **Gamma** (30–45 Hz) — local cortical processing; may be visible in
  auditory cortex sensors following stimulus onset


In [ ]:
# plot_topomap() collapses power across a frequency band for each sensor.
# bands= accepts a dict of {label: (fmin, fmax)} pairs.
# We define five standard bands that span the range of our PSD (1–100 Hz).
# Upper limit of gamma is set to 45 Hz to stay clear of the 60 Hz line noise floor.
freq_bands = {
    'Delta\n(1–4 Hz)'   : (1,  4),
    'Theta\n(4–8 Hz)'   : (4,  8),
    'Alpha\n(8–12 Hz)'  : (8,  12),
    'Beta\n(15–30 Hz)'  : (15, 30),
    'Gamma\n(30–45 Hz)' : (30, 45),
}

fig = psd.plot_topomap(
    bands=freq_bands,
    ch_type='mag',
    normalize=False,
    show=False,
)
plt.show()


### 2c. Mean vs median PSD

`compute_psd()` supports the `average` keyword, which controls how the
individual windowed periodograms within each epoch are combined:

- `average='mean'` (default) — arithmetic mean across windows. Fast and
  standard, but can be pulled upward by a single high-power window (e.g.
  a movement artifact).
- `average='median'` — median across windows, corrected for bias relative
  to the mean. More robust to outlier windows — a good choice when artifact
  rejection is incomplete.

**On clean data the two lines will overlap almost exactly** — that is the
expected and correct result. When epochs are well-cleaned (as ours are after
ICA and rejection), the mean and median of the Welch windows within each epoch
agree closely. Averaging over all epochs and channels then removes any
remaining small differences.

To make any differences visible we also plot the **ratio** (median / mean) on
a separate panel. A flat line at 1.0 confirms the two are equivalent; dips
below 1.0 would indicate frequencies where high-power outlier windows inflated
the mean relative to the median.


In [ ]:
psd_mean   = epochs_z.compute_psd(method='welch', fmin=1.0, fmax=100.0,
                                    picks='mag', average='mean',   verbose=False)
psd_median = epochs_z.compute_psd(method='welch', fmin=1.0, fmax=100.0,
                                    picks='mag', average='median', verbose=False)

# Collapse across epochs and channels for a single summary trace per estimator.
# get_data() shape: (n_epochs, n_channels, n_freqs)
mean_power   = psd_mean.get_data().mean(axis=(0, 1))
median_power = psd_median.get_data().mean(axis=(0, 1))
freqs        = psd_mean.freqs

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

# Top panel: both PSD estimates on the same axes.
# On clean data the lines will overlap — that is the expected result.
axes[0].semilogy(freqs, mean_power,   lw=1.5, label='mean')
axes[0].semilogy(freqs, median_power, lw=1.5, label='median', ls='--')
axes[0].set_ylabel('Power (T²/Hz)')
axes[0].set_title('PSD — mean vs median (Z channels, all epochs)')
axes[0].legend()

# Bottom panel: ratio median / mean.
# A flat line at 1.0 means the two estimators agree completely.
# Dips below 1.0 indicate frequencies where high-power outlier windows
# inflated the mean — i.e. residual artifacts at those frequencies.
ratio = median_power / mean_power
axes[1].axhline(1.0, color='k', lw=0.8, ls=':')
axes[1].plot(freqs, ratio, lw=1.5, color='C2')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Ratio (median / mean)')
axes[1].set_title('Ratio — values near 1.0 confirm clean data')

plt.tight_layout()
plt.show()


## 3. Time-Frequency Representations (TFR)

A **time-frequency representation** shows how power at each frequency evolves
over time. Unlike the PSD, which collapses across the entire epoch, a TFR
preserves the temporal dimension and lets you ask: *does gamma power increase
after stimulus onset? Does alpha power decrease?*

We use **Morlet wavelets** — sinusoids tapered by a Gaussian envelope. Each
wavelet has a center frequency and a number of cycles (`n_cycles`) that controls
the trade-off between time and frequency resolution:
- **More cycles** → better frequency resolution, poorer time resolution
- **Fewer cycles** → better time resolution, poorer frequency resolution

`mne.time_frequency.tfr_morlet()` returns two objects:
- **`power`** — induced power: the average of per-trial power (captures both
  phase-locked and non-phase-locked activity)
- **`itc`** — inter-trial coherence: phase consistency across trials (ranges
  0–1; values near 1 indicate a strong phase-locked response)

Other available methods: `tfr_multitaper()` (better spectral concentration),
`tfr_stockwell()` (adaptive time-frequency resolution).


In [ ]:
# Define the frequency range and resolution.
#
# Lower bound — 4 Hz:
# The data were high-passed at 0.1 Hz during preprocessing, so low frequencies
# are present in the signal. However, the TFR cannot estimate them reliably
# regardless of the filter. With n_cycles = freqs / 2, a 4 Hz wavelet is
# 2 cycles × 250 ms = 500 ms long. The usable post-stimulus window is ~800 ms
# (0 to +800 ms). Going below 4 Hz means the wavelet duration starts competing
# with the available window, causing edge effects at epoch boundaries. The
# 0.1 Hz high-pass is irrelevant here — this is purely a time-resolution limit.
#
# Upper bound — 40 Hz:
# US line noise is 60 Hz. Morlet wavelets have spectral spread: at 40 Hz with
# 20 cycles the wavelet's frequency resolution is roughly ±4 Hz, so stopping
# at 40 Hz keeps the upper edge of the highest-frequency wavelet well clear of
# the 60 Hz notch and avoids line noise leakage into the TFR estimates.
freqs    = np.arange(4, 41, 1)     # 4–40 Hz in 1 Hz steps
n_cycles = freqs / 2.0             # n_cycles scales with frequency → constant time resolution

# tfr_morlet() computes the TFR for each epoch then averages.
# return_itc=True also computes inter-trial coherence.
# decim=2 downsamples the time axis by a factor of 2 to reduce memory use
# (400 Hz → 200 Hz effective temporal resolution in the TFR).
# use_fft=True uses FFT-based convolution — faster for long signals.
power, itc = mne.time_frequency.tfr_morlet(
    epochs_z,
    freqs=freqs,
    n_cycles=n_cycles,
    use_fft=True,
    return_itc=True,
    decim=2,
    n_jobs=1,
    verbose=False,
)

print(f'Power shape : {power.data.shape}  (channels × freqs × times)')
print(f'ITC shape   : {itc.data.shape}')
print(f'Frequencies : {power.freqs[0]:.0f} – {power.freqs[-1]:.0f} Hz')
print(f'Times       : {power.times[0]*1000:.0f} – {power.times[-1]*1000:.0f} ms')


## 4. Visualising Power

### 4a. Interactive topo plot

`power.plot_topo()` shows a miniature time-frequency plot at each sensor
position. In `%matplotlib qt` mode you can click any sensor to expand it.
This is a good first pass to identify which sensors show the strongest
time-frequency structure.


In [ ]:
# plot_topo() — one TFR panel per sensor in topographical layout.
# baseline normalizes relative to a pre-stimulus window.
# mode='logratio' expresses power as log(power / baseline_power) — zero means
# no change from baseline; positive = power increase; negative = decrease.
power.plot_topo(
    baseline=(-0.400, -0.050),
    mode='logratio',
    title='Induced power — Z channels (log ratio to baseline)',
    show=False,
)
plt.show()


### 4b. Joint plot

`power.plot_joint()` combines a time-frequency plot averaged across sensors
with a row of topomaps at selected time-frequency points — similar in spirit
to `evoked.plot_joint()`. It gives a compact overview of when, at what
frequency, and where power changes occur.


In [ ]:
# plot_joint() — average TFR across channels + topomaps at selected times.
# timefreqs= specifies (time_s, freq_Hz) pairs for the topomap insets.
# We mark the stimulus onset region and the MMN window.
power.plot_joint(
    baseline=(-0.400, -0.050),
    mode='logratio',
    tmin=-0.200,
    tmax=0.600,
    timefreqs=[(0.100, 10), (0.200, 10), (0.100, 20), (0.200, 20)],
    show=False,
)
plt.show()


### 4c. Single-channel TFR

For a closer look, we can plot the full time-frequency map for the top MMN
channel identified in notebook 2.2. This shows the full frequency × time
landscape for that sensor.


In [ ]:
# Plot the TFR for a single channel.
# Replace 'ch_name' with the top MMN channel from notebook 2.2 Section 8.
ch_name = 'P5 36 Z'  # top MMN channel from notebook 2.2 Section 8

power.plot(
    picks=[ch_name],
    baseline=(-0.400, -0.050),
    mode='logratio',
    title=f'Induced power — {ch_name}',
    show=False,
)
plt.show()


## 5. Inter-Trial Coherence (ITC)

**Inter-trial coherence** (ITC) measures the consistency of the oscillatory
phase across trials at each time-frequency point. It ranges from 0 to 1:

- **ITC = 0** — phases are uniformly distributed (random); no phase locking
- **ITC = 1** — all trials have identical phase; perfect phase locking

ITC is high at frequencies and latencies where the brain response is
**phase-locked** to the stimulus — this is roughly equivalent to what the
evoked response captures, but expressed in the frequency domain. Induced
power (Section 4) without corresponding ITC indicates **non-phase-locked**
oscillatory activity (e.g. a gamma burst that occurs reliably but at variable
latency across trials).

ITC does **not** require baseline normalization — it is already a bounded
measure between 0 and 1.


In [ ]:
# itc.plot_topo() — same layout as power.plot_topo().
# No baseline needed — ITC is already on a 0–1 scale.
itc.plot_topo(
    title='Inter-trial coherence — Z channels',
    show=False,
)
plt.show()


In [ ]:
# Single-channel ITC for the same channel as above.
# High ITC in the early post-stimulus window reflects the phase-locked
# auditory response (analogous to the M100/MMN in the evoked).
itc.plot(
    picks=[ch_name],
    title=f'ITC — {ch_name}',
    show=False,
)
plt.show()


## 6. Condition Comparison: Context-matched Standard vs Deviant

We compute separate TFRs for each condition and subtract them to reveal
frequency-specific differences. As in notebook 2.1, we use **context-matched
standards** — the standard trial immediately preceding each deviant — rather
than the full ~820-trial standard set. This controls for sequential context:
both conditions share the same preceding auditory history, so any difference
reflects the mismatch response rather than differences in local adaptation.

We first identify the matching standard indices using the same `epochs.events`
logic as notebook 2.1, then select those epochs from `epochs_z`.


In [ ]:
# ── Select context-matched standard epochs ───────────────────────────────────
# epochs_z.events is a (n_epochs, 3) array: [sample_onset, 0, event_id]
# We walk the event list and collect the index of each standard trial that
# immediately precedes a deviant — identical logic to notebook 2.1 Section 8d.
std_id    = epochs_z.event_id[STANDARD_COND]
dev_id    = epochs_z.event_id[DEVIANT_COND]
event_ids = epochs_z.events[:, 2]

preceding_std_indices = [
    i - 1
    for i, ev_id in enumerate(event_ids)
    if ev_id == dev_id and i > 0 and event_ids[i - 1] == std_id
]

print(f'Deviants found            : {(event_ids == dev_id).sum()}')
print(f'Preceding standards found : {len(preceding_std_indices)}')

epochs_z_std_matched = epochs_z[preceding_std_indices]
epochs_z_dev         = epochs_z[DEVIANT_COND]

# ── Compute TFRs ─────────────────────────────────────────────────────────────
# return_itc=False returns a single AverageTFR object (not a tuple) — no unpacking.
power_std = mne.time_frequency.tfr_morlet(
    epochs_z_std_matched, freqs=freqs, n_cycles=n_cycles,
    use_fft=True, return_itc=False, decim=2, n_jobs=1, verbose=False,
)
power_dev = mne.time_frequency.tfr_morlet(
    epochs_z_dev, freqs=freqs, n_cycles=n_cycles,
    use_fft=True, return_itc=False, decim=2, n_jobs=1, verbose=False,
)

# Apply baseline to each before subtracting so both are on the same scale.
power_std.apply_baseline(baseline=(-0.400, -0.050), mode='logratio')
power_dev.apply_baseline(baseline=(-0.400, -0.050), mode='logratio')

# Difference: deviant − context-matched standard.
power_diff = power_dev.copy()
power_diff.data -= power_std.data
power_diff.comment = 'Deviant − context-matched standard (log ratio to baseline)'

print(f'\nContext-matched standard nave : {power_std.nave}')
print(f'Deviant nave                  : {power_dev.nave}')
print(f'Trial counts match            : {power_std.nave == power_dev.nave}')


In [ ]:
# Plot the difference TFR — joint plot with topomaps.
power_diff.plot_joint(
    tmin=-0.200,
    tmax=0.600,
    timefreqs=[(0.200, 10), (0.200, 20)],
    title='Power difference: deviant − context-matched standard (log ratio to baseline)',
    show=False,
)
plt.show()


### 6c. Per-condition ITC: standard vs deviant

In section 5 we computed ITC across all trials pooled together. Here we compute
it separately for context-matched standard and deviant trials and compare them.

Recall that ITC measures **phase consistency across trials** at each
time-frequency point — it is high wherever the brain responds with a
reproducible phase relationship to the stimulus, regardless of power.

Comparing ITC per condition in the MMN window (150–250 ms) can reveal whether
the mismatch response is partly a **phase-locking phenomenon**: if deviants
produce higher ITC than context-matched standards in that window, it suggests
that the brain's response to unexpected sounds is not just stronger but also
more temporally consistent across trials.

> **Note on trial counts:** Both conditions here are 166 trials (context-matched
> sets). TFR estimates — and especially ITC — are noisier with fewer trials, so
> interpret these plots with more caution than the ~820-trial full-standard results
> from section 5.


In [ ]:
# Compute ITC separately for each condition.
# return_itc=True returns a (power, itc) tuple — we discard power with _.
# We reuse epochs_z_std_matched and epochs_z_dev selected above.
_, itc_std = mne.time_frequency.tfr_morlet(
    epochs_z_std_matched, freqs=freqs, n_cycles=n_cycles,
    use_fft=True, return_itc=True, decim=2, n_jobs=1, verbose=False,
)
_, itc_dev = mne.time_frequency.tfr_morlet(
    epochs_z_dev, freqs=freqs, n_cycles=n_cycles,
    use_fft=True, return_itc=True, decim=2, n_jobs=1, verbose=False,
)

print(f'ITC context-matched standard nave : {itc_std.nave}')
print(f'ITC deviant nave                  : {itc_dev.nave}')


In [ ]:
# Topo layout of ITC for each condition — one panel per sensor.
# No baseline needed; ITC is already on a 0–1 scale.
itc_std.plot_topo(
    title=f'ITC — context-matched standard (nave={itc_std.nave})',
    show=False,
)
plt.show()

itc_dev.plot_topo(
    title=f'ITC — deviant (nave={itc_dev.nave})',
    show=False,
)
plt.show()


In [ ]:
# ITC difference: deviant − standard.
# ITC values are bounded 0–1, so the difference ranges from −1 to +1.
# Positive values mean deviants produced stronger phase locking than standards.
itc_diff = itc_dev.copy()
itc_diff.data -= itc_std.data
itc_diff.comment = 'ITC: deviant − standard'

# Joint plot: average ITC difference across channels + topomaps at key windows.
# The MMN window (150–250 ms) and the early M100 window (75–125 ms) are marked.
itc_diff.plot_joint(
    tmin=-0.200,
    tmax=0.600,
    timefreqs=[(0.100, 8), (0.200, 8), (0.100, 4), (0.200, 4)],
    title='ITC difference: deviant − context-matched standard',
    show=False,
)
plt.show()


In [ ]:
# Single-channel ITC difference at a user-selected sensor.
# Set ch_name to your top MMN channel (identified in notebook 2.2 Section 8).
ch_name = 'P5 36 Z'  # <-- replace with your channel

itc_diff.plot(
    picks=[ch_name],
    title=f'ITC difference: deviant − context-matched standard — {ch_name}',
    show=False,
)
plt.show()


### Interpreting low-frequency ITC differences

> ⚠️ **If the largest ITC difference appears at ~4 Hz between 100–225 ms, that
> is the expected result — but it requires careful interpretation.**

If you compare the three time-frequency plots from sections 4c, 5, and 6c,
you may notice that they all share a similar onset ( ~100 ms) and peak latency
( ~180 ms), but differ in their dominant frequency:

- **4c — single-channel power (~6–7 Hz):** the amplitude-weighted center of
  the auditory response pooled across all trials
- **5 — single-channel ITC (~8–9 Hz):** the phase-locking of the same
  response; slightly higher in frequency because ITC is not biased by
  amplitude, so it tracks the sharpness of phase alignment rather than
  the power envelope
- **6c — ITC difference (~4 Hz):** the deviant-specific excess phase locking
  sits at a lower frequency

One possible interpretation: since the timing appears consistent across all three
plots, the frequency axis may not be separating early from late neural
components — it may instead be separating a **shared fast response** (present
for both conditions, dominant at 6–9 Hz) from a **deviant-specific slower
modulation** (the MMN, dominant at ~4 Hz). Both are centered around the same
latency, but the MMN is a broader, slower deflection riding on top of the
shared M100 — which would push its TFR signature to lower frequencies even
though it peaks at the same time.

However, this interpretation should be treated with caution. As noted above,
`tfr_morlet()` measures **total** activity — the phase-locked evoked response
and any induced oscillations are mixed together. The low-frequency ITC
difference could equally reflect the MMN evoked response itself rather than
a distinct oscillatory phenomenon. To separate the two, the evoked response
would need to be subtracted from each epoch before computing the TFR (see
the induced-activity note below).

**To isolate induced activity**, subtract the condition-average evoked response
from each epoch *before* computing the TFR:

```python
# Subtract the evoked response from each epoch to isolate induced activity.
# This removes the phase-locked component, leaving only non-phase-locked oscillations.
epochs_z_dev_induced = epochs_z_dev.copy()
epochs_z_dev_induced._data -= power_dev_evoked.data[np.newaxis, :, :]  # evoked shape: (ch, time)
```

What remains after subtraction is purely **induced** activity — oscillations
that occur reliably across trials but without a fixed phase relationship to the
stimulus. Comparing induced power between conditions then isolates true
oscillatory dynamics from the evoked response. This is covered in more advanced
TFR analyses and is standard practice when making claims about condition
differences in neural oscillations.


## 7. Summary

| Task | Method |
|------|--------|
| Compute PSD from epochs | `epochs.compute_psd(method='welch', fmin=..., fmax=...)` |
| Plot spectrum | `psd.plot()` |
| Spatial PSD distribution | `psd.plot_topomap()` |
| Robust PSD estimate | `epochs.compute_psd(average='median')` |
| Compute TFR (Morlet) | `mne.time_frequency.tfr_morlet(epochs, freqs, n_cycles)` |
| Topo layout of TFR | `power.plot_topo(baseline=..., mode='logratio')` |
| Joint TFR + topomaps | `power.plot_joint(timefreqs=[(t, f), ...])` |
| Single-channel TFR | `power.plot(picks=[ch_name])` |
| Inter-trial coherence | returned as second output of `tfr_morlet(..., return_itc=True)` |
| ITC topo layout | `itc.plot_topo()` |
| Baseline normalization | `power.apply_baseline(baseline=(...), mode='logratio')` |
| Condition difference TFR | Subtract `.data` arrays after baseline normalization |

### Key concepts

- **PSD** collapses across time — useful for characterising the overall spectral
  profile of the recording (noise floor, line noise, oscillations)
- **TFR** preserves time — shows when oscillatory power changes relative to the stimulus
- **`n_cycles / freq`** sets the wavelet duration in seconds; scaling `n_cycles` with
  frequency gives constant time resolution across the frequency axis
- **`mode='logratio'`** expresses power relative to baseline in log units — zero means
  no change, positive means increase, negative means decrease
- **ITC** measures phase locking (0–1); high ITC without high power indicates a
  phase-locked response that cancels in power due to phase variability across trials
- **Induced vs evoked power**: induced power includes both phase-locked and
  non-phase-locked activity; subtracting the evoked response before computing TFR
  isolates purely non-phase-locked (induced) activity

### Key OPM-MEG reminders

- Always restrict to Z channels (`endswith(' Z')`) for TFR and PSD —
  X and Y channels have different spatial profiles and mixing them adds noise
- The 400 Hz sampling rate limits analysis to frequencies below 200 Hz (Nyquist);
  in practice OPM noise dominates above ~100 Hz
- `decim=2` in `tfr_morlet` is recommended to keep memory use manageable;
  it reduces the TFR time axis to 200 Hz effective rate, sufficient for responses
  in the 4–40 Hz range

### Next steps
- **Source localisation** — forward model and inverse solution (MNE, dSPM, beamformer)
